# FraudBench Experiment Runner (Colab)

This notebook runs FraudBench experiments using Colab GPU.

**Workflow:**
1. Mount Google Drive for data and results persistence
2. Clone/update the repo
3. Install dependencies
4. Link datasets from Google Drive
5. Run experiments via config files
6. Results auto-save to Google Drive

In [ ]:
# Cell 1: Verify GPU is available
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Cell 2: Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/FraudBench"
for subdir in ["data", "results", "models", "logs"]:
    os.makedirs(os.path.join(DRIVE_ROOT, subdir), exist_ok=True)
    print(f"  {DRIVE_ROOT}/{subdir}/")

print("\nGoogle Drive mounted and directories ready.")

In [ ]:
# Cell 3: Clone or update repo
import os

REPO_URL = "https://github.com/YOUR_USERNAME/Capstone_FraudBench.git"  # UPDATE THIS
REPO_DIR = "/content/Capstone_FraudBench"

if os.path.exists(REPO_DIR):
    print("Repo exists. Pulling latest changes...")
    os.chdir(REPO_DIR)
    !git pull
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"\nWorking directory: {os.getcwd()}")
!git log --oneline -3

In [ ]:
# Cell 4: Install dependencies
# Colab does not have uv, so we use pip install -e . (editable mode)
!pip install -e . -q 2>&1 | tail -5

# If the above fails, fall back to the flat requirements file:
# !pip install -r requirements-colab.txt -q

In [ ]:
# Cell 5: Symlink dataset directories from Google Drive
# The loader expects data at <project_root>/datasets/<DatasetName>/
# We symlink individual dirs (not the whole datasets/ folder,
# because it also contains Python source files like loader.py)
import os

DRIVE_DATA = "/content/drive/MyDrive/FraudBench/data"
DATASETS_DIR = "/content/Capstone_FraudBench/datasets"

for dataset_dir in ["CCFD", "ieee-fraud-detection", "LCLD", "Sparkov"]:
    src = os.path.join(DRIVE_DATA, dataset_dir)
    dst = os.path.join(DATASETS_DIR, dataset_dir)
    if os.path.islink(dst):
        os.unlink(dst)
    if os.path.exists(src):
        os.symlink(src, dst)
        files = os.listdir(src)
        print(f"  Linked: {dataset_dir}/ ({len(files)} files)")
    else:
        print(f"  NOT FOUND on Drive: {dataset_dir}/ -- upload to {src}")

print("\nDataset symlinks ready.")

In [ ]:
# Cell 6: Run a single experiment
# Change CONFIG to run different experiments.
# Available configs: ls configs/

CONFIG = "configs/ccfd_baseline.yaml"

!python -m runner.run --config {CONFIG}

In [ ]:
# Cell 7: Batch runner -- run multiple experiments sequentially
import subprocess
import time

configs = [
    "configs/ccfd_baseline.yaml",
    "configs/ccfd_adv_train.yaml",
    "configs/ccfd_input_val.yaml",
    "configs/ccfd_tree.yaml",
    # Add more configs as needed -- see configs/ for all options
]

for i, config in enumerate(configs):
    sep = '=' * 60
    print(f'\n{sep}')
    print(f"Experiment {i+1}/{len(configs)}: {config}")
    print(sep)

    start = time.time()
    result = subprocess.run(
        ["python", "-m", "runner.run", "--config", config],
        capture_output=True, text=True
    )
    elapsed = time.time() - start

    if result.returncode == 0:
        print(f"SUCCESS ({elapsed:.1f}s)")
        for line in result.stdout.strip().split('\n')[-5:]:
            print(f"  {line}")
    else:
        print(f"FAILED ({elapsed:.1f}s)")
        print(result.stderr[-500:] if result.stderr else "No error output")

sep = '=' * 60
print(f'\n{sep}')
print("All experiments complete. Run the next cell to save results to Drive.")

In [ ]:
# Cell 8: Copy results to Google Drive for persistence
import shutil
import glob
import os

RESULTS_SRC = "/content/Capstone_FraudBench/results"
RESULTS_DST = "/content/drive/MyDrive/FraudBench/results"

for f in glob.glob(os.path.join(RESULTS_SRC, "*")):
    if os.path.isfile(f):
        dst = os.path.join(RESULTS_DST, os.path.basename(f))
        shutil.copy2(f, dst)
        print(f"Saved: {os.path.basename(f)}")

# Also copy figures if they exist
figures_src = os.path.join(RESULTS_SRC, "figures")
figures_dst = os.path.join(RESULTS_DST, "figures")
if os.path.isdir(figures_src):
    if os.path.exists(figures_dst):
        shutil.rmtree(figures_dst)
    shutil.copytree(figures_src, figures_dst)
    print("Saved: figures/ directory")

print(f"\nAll results backed up to {RESULTS_DST}")

In [ ]:
# Cell 9: Run all seeds (full benchmark)
# WARNING: This runs ALL experiments (52 configs x 3 seeds = 156 total).
# Estimated time: 2-4 hours on T4 GPU.
# Only run this when ready for a full benchmark pass.

# !python scripts/run_all_seeds.py

In [ ]:
# Cell 10: Session cleanup
# IMPORTANT: Run this when done to stop consuming compute units!
# First make sure results are saved (run Cell 8 first!)

print("Make sure you have saved results to Google Drive (Cell 8) before disconnecting!")
print("To disconnect: Runtime > Disconnect and delete runtime")
print("Or uncomment the line below:")
# from google.colab import runtime; runtime.unassign()